# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and set your `OPENAI_API_KEY` value (in Cell 1). Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1 — Install dependencies
# Run this cell first, then RESTART THE RUNTIME (Runtime → Restart runtime),
# then run Cell 2 to launch the UI.

!pip install -q \
  openai \
  langchain-text-splitters \
  pypdf \
  ipywidgets

# Enable ipywidgets in Colab
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

# Quick version check
import importlib.metadata as _m
for _p in ["openai", "langchain-text-splitters", "pypdf", "ipywidgets"]:
    try:    print(f"  {_p}=={_m.version(_p)}")
    except: print(f"  {_p} not found")

print("\nDone. Restart runtime now, then run Cell 2.")


In [ ]:
# Colab Cell 2 — RAG UI
# Run after Cell 1 (restart runtime first if Cell 1 just finished).

import io, re, html as _html, threading
import numpy as np
import requests as _req          # pre-installed in Colab; pure sockets, no asyncio
from pypdf import PdfReader
from IPython.display import display
import ipywidgets as widgets

try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Shared state ──────────────────────────────────────────────────────────────
_state = {
    "chunks":        [],
    "embeddings":    [],
    "indexed_files": [],
}
_indexing = threading.Event()   # set while indexing is in progress

# ── OpenAI via raw requests — works reliably from any thread ──────────────────

def _key():
    return api_key_input.value.strip()

def _embed(texts: list) -> list:
    r = _req.post(
        "https://api.openai.com/v1/embeddings",
        headers={"Authorization": f"Bearer {_key()}"},
        json={"model": "text-embedding-3-small", "input": texts},
        timeout=120,
    )
    r.raise_for_status()
    return [d["embedding"] for d in r.json()["data"]]

def _chat(prompt: str) -> str:
    r = _req.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {_key()}"},
        json={
            "model": "gpt-4o-mini",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
        },
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

# ── PDF extraction ────────────────────────────────────────────────────────────

def _pdf_to_text(b: bytes) -> str:
    reader = PdfReader(io.BytesIO(b))
    pages = []
    for p in reader.pages:
        try:    pages.append(p.extract_text() or "")
        except: pages.append("")
    return "\n\n".join(pages)

_SPLITTER = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# ── Indexing ──────────────────────────────────────────────────────────────────

def index_files(files: list):
    for fname, pdf_bytes in files:
        _set_status(f"Reading '{fname}'…", "#aaa")
        text   = _pdf_to_text(pdf_bytes)
        chunks = _SPLITTER.split_text(text)
        if not chunks:
            _set_status(f"No text found in '{fname}' — skipped.", "orange")
            continue
        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i : i + batch_size]
            _set_status(f"Embedding {i+1}–{i+len(batch)}/{len(chunks)} chunks…", "#aaa")
            embs = _embed(batch)
            _state["chunks"].extend(batch)
            _state["embeddings"].extend(embs)
        if fname not in _state["indexed_files"]:
            _state["indexed_files"].append(fname)
        _refresh_indexed()

# ── Retrieval (numpy cosine similarity — no DB, fully thread-safe) ─────────────

def _retrieve(query: str, n: int = 5) -> list:
    if not _state["embeddings"]:
        return []
    q   = np.array(_embed([query])[0], dtype=np.float32)
    mat = np.array(_state["embeddings"],  dtype=np.float32)
    sim = (mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-10)) @ \
          (q   / (np.linalg.norm(q)                            + 1e-10))
    idx = np.argsort(sim)[-min(n, len(sim)):][::-1]
    return [_state["chunks"][i] for i in idx]

# ── Prompts ───────────────────────────────────────────────────────────────────

_NOT_FOUND = "I cannot find that information in the provided document."

_QA_TMPL = (
    "You are a helpful assistant specialising in {domain}.\n"
    "Answer ONLY using the CONTEXT below.\n"
    "If the answer is absent, reply EXACTLY: \"{not_found}\"\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n\nAnswer:"
)
_SUM_TMPL = (
    "You are a summarisation assistant specialising in {domain}.\n"
    "Produce a clear structured summary with headings from the CONTEXT below.\n\n"
    "CONTEXT:\n{context}"
)

# ── RAG pipeline ──────────────────────────────────────────────────────────────

def query_with_rag(question: str, domain: str) -> str:
    chunks  = _retrieve(question)
    context = "\n\n".join(chunks) if chunks else ""
    answer  = _chat(_QA_TMPL.format(domain=domain, context=context,
                                     question=question, not_found=_NOT_FOUND))
    if _NOT_FOUND not in answer:
        return answer
    fb = _chat(f"Question: {question}\n\n"
               "The document didn't contain this. Answer from general knowledge.")
    return "⚠️ **Not in document** — from general knowledge:\n\n" + fb

def generate_summary(domain: str) -> str:
    chunks  = _retrieve("summary overview key points introduction conclusion", n=8)
    context = "\n\n".join(chunks) if chunks else ""
    return _chat(_SUM_TMPL.format(domain=domain, context=context))

# ── Markdown → HTML ───────────────────────────────────────────────────────────

def _md(text: str) -> str:
    out = []
    for line in text.split("\n"):
        if   line.startswith("### "): out.append(f"<h4 style='margin:6px 0 2px'>{_html.escape(line[4:])}</h4>")
        elif line.startswith("## "):  out.append(f"<h3 style='margin:8px 0 2px'>{_html.escape(line[3:])}</h3>")
        elif line.startswith("# "):   out.append(f"<h2 style='margin:10px 0 2px'>{_html.escape(line[2:])}</h2>")
        elif line.startswith(("- ","* ")): out.append(f"<li style='margin:2px 0'>{_html.escape(line[2:])}</li>")
        elif not line.strip():        out.append("<br>")
        else:
            s = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>",
                re.sub(r"\*(.+?)\*", r"<i>\1</i>", _html.escape(line)))
            out.append(f"<p style='margin:2px 0'>{s}</p>")
    return "".join(out)

# ── Widgets ───────────────────────────────────────────────────────────────────

api_key_input  = widgets.Password(placeholder="sk-…", description="OpenAI Key:",
                                   layout=widgets.Layout(width="380px"))
api_key_status = widgets.HTML('<span style="color:orange">Enter your key above</span>')

def _on_key(change):
    k = change["new"].strip()
    api_key_status.value = (
        '<span style="color:lightgreen">&#10003; Key ready</span>'
        if k.startswith("sk-") and len(k) > 20
        else '<span style="color:orange">Needs a valid sk-… key</span>'
    )
api_key_input.observe(_on_key, names="value")

_DEFAULTS  = ["Media", "Law", "Telecom", "General"]
domain_dd  = widgets.Dropdown(options=_DEFAULTS, value="General",
                               description="Category:", layout=widgets.Layout(width="220px"))
new_cat    = widgets.Text(placeholder="e.g. Finance…", layout=widgets.Layout(width="200px"))
add_cat    = widgets.Button(description="+ Add",    button_style="warning", layout=widgets.Layout(width="72px"))
rm_cat     = widgets.Button(description="✕ Remove", button_style="danger",  layout=widgets.Layout(width="90px"))
cat_msg    = widgets.HTML("")

def _do_add(b):
    nm = new_cat.value.strip()
    if not nm:         cat_msg.value = '<span style="color:orange">Type a name.</span>'; return
    opts = list(domain_dd.options)
    if nm in opts:     cat_msg.value = f'<span style="color:orange">"{nm}" already exists.</span>'; return
    opts.append(nm); domain_dd.options = opts; domain_dd.value = nm; new_cat.value = ""
    cat_msg.value = f'<span style="color:lightgreen">Added "{nm}"</span>'
add_cat.on_click(_do_add)

def _do_rm(b):
    cur = domain_dd.value
    if cur in _DEFAULTS: cat_msg.value = f'<span style="color:orange">Cannot remove built-in "{cur}".</span>'; return
    opts = [o for o in domain_dd.options if o != cur]
    domain_dd.options = opts; domain_dd.value = opts[-1] if opts else None
    cat_msg.value = f'<span style="color:lightgreen">Removed "{cur}"</span>'
rm_cat.on_click(_do_rm)

upload_out = widgets.Output(layout=widgets.Layout(border="1px dashed #555",
                             padding="4px", min_height="24px", margin="4px 0"))
upload_btn = widgets.Button(description="📂 Upload PDF(s)", button_style="info",
                             layout=widgets.Layout(width="160px"))
clear_btn  = widgets.Button(description="Clear all", button_style="danger",
                             layout=widgets.Layout(width="90px"))

# indexed_html is at TOP LEVEL of the final VBox so widget updates from threads render
indexed_html = widgets.HTML('<i style="color:#888">No documents indexed yet.</i>')
status_html  = widgets.HTML('<i style="color:#888">Ready.</i>')
answer_html  = widgets.HTML('<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>')

def _refresh_indexed():
    files = _state["indexed_files"]
    if not files:
        indexed_html.value = '<i style="color:#888">No documents indexed yet.</i>'
        return
    rows = "".join(f'<li style="color:lightgreen">&#10003; {_html.escape(f)}</li>' for f in files)
    indexed_html.value = (
        f'<b style="color:#ccc">Indexed ({len(files)} doc{"s" if len(files)!=1 else ""}):</b>'
        f'<ul style="margin:2px 0;padding-left:16px">{rows}</ul>'
    )

def _set_status(msg, color="#ccc"):
    status_html.value = f'<span style="color:{color}">{_html.escape(str(msg))}</span>'

def _set_answer(text):
    answer_html.value = (
        '<div style="font-family:sans-serif;font-size:14px;line-height:1.7;'
        'padding:10px;border:1px solid #444;border-radius:4px">'
        + _md(text) + '</div>'
    )

# ── Upload ────────────────────────────────────────────────────────────────────

def on_upload(b):
    if not _key(): _set_status("Enter your OpenAI API key first.", "orange"); return
    upload_out.clear_output()
    try:
        from google.colab import files as _gf
    except ImportError:
        with upload_out: print("Not running in Colab."); return

    with upload_out:
        print("Opening file picker — select one or more PDFs…")
        try:    raw = _gf.upload()
        except Exception as ex: print("Upload error:", ex); return
    upload_out.clear_output()

    if not raw: _set_status("No files uploaded.", "#aaa"); return
    pdfs    = [(fn, bytes(c)) for fn, c in raw.items() if fn.lower().endswith(".pdf")]
    skipped = [fn for fn in raw if not fn.lower().endswith(".pdf")]
    if skipped: _set_status(f"Skipped non-PDF: {', '.join(skipped)}", "orange")
    if not pdfs: _set_status("No PDF files — please upload .pdf files.", "orange"); return

    def _worker():
        _indexing.set()
        try:
            index_files(pdfs)
            _refresh_indexed()
            n = len(_state["indexed_files"])
            _set_status(f"✓ {n} doc(s) ready. Ask a question or generate a summary.", "lightgreen")
        except Exception as ex:
            _set_status(f"Indexing failed: {ex}", "tomato")
        finally:
            _indexing.clear()
    threading.Thread(target=_worker, daemon=True).start()
upload_btn.on_click(on_upload)

def on_clear(b):
    if _indexing.is_set(): _set_status("Wait for indexing to finish first.", "orange"); return
    _state.update({"chunks": [], "embeddings": [], "indexed_files": []})
    _refresh_indexed()
    answer_html.value = '<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>'
    _set_status("Cleared. Upload new PDF(s) to start again.", "#aaa")
clear_btn.on_click(on_clear)

# ── Ask ───────────────────────────────────────────────────────────────────────

ask_text = widgets.Text(description="Question:", layout=widgets.Layout(width="65%"))
ask_btn  = widgets.Button(description="Ask", button_style="primary",
                           layout=widgets.Layout(width="80px"))

def on_ask(b):
    if not _key():                   _set_status("Enter your OpenAI API key first.", "orange"); return
    if _indexing.is_set():           _set_status("Still indexing — wait for the green ✓ status.", "orange"); return
    if not _state["embeddings"]:     _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    q = ask_text.value.strip()
    if not q:                        _set_status("Type a question first.", "orange"); return

    ask_btn.disabled = True
    n = len(_state["indexed_files"])
    _set_status(f"Searching {n} doc(s)…", "#aaa")
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Thinking…</i></div>'

    def _worker():
        try:
            ans = query_with_rag(q, domain_dd.value)
            _set_answer(ans)
            _set_status("Done ✓", "lightgreen")
        except Exception as ex:
            _set_status(f"Query failed: {ex}", "tomato")
            answer_html.value = f'<div style="color:tomato;padding:8px"><b>Error:</b> {_html.escape(str(ex))}</div>'
        finally:
            ask_btn.disabled = False
    threading.Thread(target=_worker, daemon=True).start()
ask_btn.on_click(on_ask)

# ── Summary ───────────────────────────────────────────────────────────────────

summary_btn = widgets.Button(description="Generate Summary", button_style="info",
                              layout=widgets.Layout(width="165px"))

def on_summary(b):
    if not _key():               _set_status("Enter your OpenAI API key first.", "orange"); return
    if _indexing.is_set():       _set_status("Still indexing — wait for the green ✓ status.", "orange"); return
    if not _state["embeddings"]: _set_status("No documents indexed. Upload a PDF first.", "orange"); return

    summary_btn.disabled = True
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Generating summary…</i></div>'

    def _worker():
        try:
            summ = generate_summary(domain_dd.value)
            _set_answer(summ)
            _set_status("Summary ready ✓", "lightgreen")
        except Exception as ex:
            _set_status(f"Summary failed: {ex}", "tomato")
            answer_html.value = f'<div style="color:tomato;padding:8px"><b>Error:</b> {_html.escape(str(ex))}</div>'
        finally:
            summary_btn.disabled = False
    threading.Thread(target=_worker, daemon=True).start()
summary_btn.on_click(on_summary)

# ── Layout ────────────────────────────────────────────────────────────────────

left_box = widgets.VBox([
    widgets.HTML("<b>Step 2 — Upload Document(s)</b>"),
    domain_dd,
    widgets.HBox([new_cat, add_cat, rm_cat]),
    cat_msg,
    widgets.HBox([upload_btn, clear_btn]),
    upload_out,
])
right_box = widgets.VBox([
    widgets.HTML("<b>Step 3 — Ask or Summarise</b>"),
    widgets.HTML('<span style="font-size:12px;color:#aaa">Wait for the green ✓ status before asking.</span>'),
    widgets.HBox([ask_text, ask_btn]),
    widgets.HTML('<span style="font-size:12px;color:#aaa;margin:4px 0;display:block">— or —</span>'),
    summary_btn,
])

ui = widgets.VBox([
    widgets.HTML("<b>Step 1 — OpenAI API Key</b>"),
    widgets.HBox([api_key_input, api_key_status]),
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HBox([left_box, right_box], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<hr style='margin:6px 0'>"),
    indexed_html,          # top-level → thread updates always render
    widgets.HTML("<b style='display:block;margin-top:4px'>Status</b>"),
    status_html,
    widgets.HTML("<b style='display:block;margin-top:6px'>Answer</b>"),
    answer_html,
], layout=widgets.Layout(padding="12px", max_width="960px"))

display(ui)
